In [1]:
# # For kaggle:
# !git clone https://github.com/maTiahK/CORT_SI.git
# %cd /kaggle/working/CORT_SI

# !python -m pip install -q --upgrade pip
# !python -m pip install -q -e .


# from cort_si import SI, SI_randj, gen_data
# print("CORT_SI import OK")

# Example 1: Computing p-values with Adaptive CoRT-SI

This notebook demonstrates how to use the `cort_si` package to compute selective inference p-values for Adaptive CoRT-SI.

In [2]:
import random
import sys

import numpy as np

sys.path.append('..')

from cort_si import SI, SI_randj, gen_data

## 1. Generate Synthetic Data

In [3]:
np.random.seed(0)
random.seed(0)

p = 300
nS = 100
nT = 50
K = 2
true_beta_scale = 1.0
rho = 0.0
sigma_noise = 1.0
source_shift_sd = 0.3
covariate_shift = False
seed = 0

XS_list, YS_list, X0, Y0, _, SigmaS_list, Sigma0, beta0 = gen_data.generate_data(
    p=p,
    nS=nS,
    nT=nT,
    K=K,
    true_beta=true_beta_scale,
    rho=rho,
    sigma_noise=sigma_noise,
    source_shift_sd=source_shift_sd,
    covariate_shift=covariate_shift,
    seed=seed,
)

print(f'Number of auxiliary groups: {len(XS_list)}')
print(f'Feature dimension: {p}')
print(f'Nonzero target coefficients: {int(np.count_nonzero(beta0))}')
print(f'Source sample size: {nS}')
print(f'Target sample size: {X0.shape[0]}')

Number of auxiliary groups: 2
Feature dimension: 300
Nonzero target coefficients: 5
Source sample size: 100
Target sample size: 50


## 2. Set Parameters

In [4]:
lambda_sel = 0.5
lambda0 = 0.5
lambdak_list = [0.5] * len(XS_list)
T = 3
verbose = True

print(f'lambda_sel: {lambda_sel}')
print(f'lambda0: {lambda0}')
print(f'T: {T}')
print('Truncation defaults: z_min=-20, z_max=20')
print(f'verbose: {verbose}')

lambda_sel: 0.5
lambda0: 0.5
T: 3
Truncation defaults: z_min=-20, z_max=20
verbose: True


## 3. Compute p-values for All Selected Features

In [7]:
p_values = SI(
    X0,
    Y0,
    XS_list,
    YS_list,
    lambda_sel=lambda_sel,
    lambda0=lambda0,
    lambdak_list=lambdak_list,
    SigmaS_list=SigmaS_list,
    Sigma0=Sigma0,
    T=T,
    verbose=verbose,
)

if p_values is not None:
    print(f'Number of selected features: {len(p_values)}')
    print('\nFeature index and p-values:')
    for j, p_val in p_values:
        print(f'Feature {j}: beta0[{j}] = {beta0[j]:.4f}, p-value = {p_val:.4f}')
else:
    print('No features selected')

KeyboardInterrupt: 

## 4. Compute a p-value for a Random Selected Feature

This section reuses the same synthetic-data setup and parameter choice, then applies `SI_randj(...)` to one randomly chosen selected target feature.

In [8]:
rng_seed = 11

if p_values is None:
    print('No features selected')
else:
    selected_features = [j for j, _ in p_values]
    j_rand = random.Random(rng_seed).choice(selected_features)
    random.seed(rng_seed)
    p_value_rand = SI_randj(
        X0,
        Y0,
        XS_list,
        YS_list,
        lambda_sel=lambda_sel,
        lambda0=lambda0,
        lambdak_list=lambdak_list,
        SigmaS_list=SigmaS_list,
        Sigma0=Sigma0,
        T=T,
        verbose=verbose,
    )

    print(f'Number of selected features: {len(selected_features)}')
    print(f'Selected feature set: {selected_features}')
    if p_value_rand is not None:
        print(f'Random selected feature: {j_rand}')
        print(f'true beta0[{j_rand}] = {beta0[j_rand]:.4f}')
        print(f'p-value = {p_value_rand:.4f}')
    else:
        print('No valid truncation interval for the random selected feature')

[SI_randj] feature 59 intervals_source = [(-2.560789101814015, -2.4971713791222254), (-2.4971613791222254, -2.4251906963240653), (-2.425180696324065, -2.353812833450646), (-2.353802833450646, -2.317271752387989), (-2.317261752387989, -2.222441085425549), (-2.222431085425549, -2.2044082221540253), (-2.2043982221540253, -1.9581446978471282), (-1.9581346978471281, -1.9536027581995177), (-1.9535927581995176, -1.7696419645249786), (-1.7696319645249785, -1.7031072480267018), (-1.7030972480267017, 1.3000706495359817)]
[SI_randj] feature 59 intervals_model = [(np.float64(0.2721035922718848), np.float64(0.4393787287533048))]
Number of selected features: 14
Selected feature set: [0, 1, 2, 3, 4, 15, 47, 59, 60, 87, 112, 140, 190, 270]
Random selected feature: 59
true beta0[59] = 0.0000
p-value = 0.4597
